In [ ]:
from rdflib import Graph, Namespace, RDF, RDFS, Literal, URIRef
from pathlib import Path

In [2]:
# Cargamos el toy KG desde el archivo Turtle
g = Graph()
g.parse("../data/kg/toy_udesa.ttl", format="turtle")

print(f"Grafo cargado: {len(g)} triples")

Grafo cargado: 64 triples


In [3]:
# Definimos el namespace para poder referirnos a recursos del grafo
# de forma cómoda desde Python (equivalente a @prefix : en Turtle)
UDESA = Namespace("http://example.org/udesa/")

In [4]:
# Recorremos todos los triples (sujeto, predicado, objeto) que digan
# "X es una Materia". O sea: todos los triples con predicado RDF.type
# y objeto UDESA.Materia
print("Materias en el grafo:")
for sujeto in g.subjects(predicate=RDF.type, object=UDESA.Materia):
    # Para cada materia, buscamos su rdfs:label
    label = g.value(subject=sujeto, predicate=RDFS.label)
    print(f"  - {sujeto} → {label}")

Materias en el grafo:
  - http://example.org/udesa/Algoritmos → Algoritmos y Estructuras de Datos
  - http://example.org/udesa/MachineLearning → Machine Learning
  - http://example.org/udesa/NLP → Procesamiento de Lenguaje Natural
  - http://example.org/udesa/BasesDatos → Bases de Datos
  - http://example.org/udesa/Microeconomia → Microeconomía


In [6]:
# Query 1 — Listar todas las materias y sus labels
query_1 = """
PREFIX :     <http://example.org/udesa/>
PREFIX rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?materia ?label
WHERE {
    ?materia rdf:type :Materia .
    ?materia rdfs:label ?label .
}
"""

print("Materias:")
for fila in g.query(query_1):
    print(f"  - {fila.label}  ({fila.materia})")

Materias:
  - Algoritmos y Estructuras de Datos  (http://example.org/udesa/Algoritmos)
  - Machine Learning  (http://example.org/udesa/MachineLearning)
  - Procesamiento de Lenguaje Natural  (http://example.org/udesa/NLP)
  - Bases de Datos  (http://example.org/udesa/BasesDatos)
  - Microeconomía  (http://example.org/udesa/Microeconomia)


In [7]:
# Query 2 — Materias de tercer año en adelante
query_2 = """
PREFIX :     <http://example.org/udesa/>
PREFIX rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?label ?año
WHERE {
    ?materia rdf:type :Materia ;
             rdfs:label ?label ;
             :año ?año .
    FILTER (?año >= 3)
}
ORDER BY ?año
"""

print("Materias de 3er año en adelante:")
for fila in g.query(query_2):
    print(f"  - {fila.label} (año {fila.año})")

Materias de 3er año en adelante:
  - Machine Learning (año 3)
  - Bases de Datos (año 3)
  - Procesamiento de Lenguaje Natural (año 4)


In [8]:
# Query 3 — Materias con info completa (con OPTIONAL)
query_3 = """
PREFIX :     <http://example.org/udesa/>
PREFIX rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?label ?año ?profLabel
WHERE {
    ?materia rdf:type :Materia ;
             rdfs:label ?label ;
             :año ?año .
    OPTIONAL {
        ?materia :dictadaPor ?prof .
        ?prof rdfs:label ?profLabel .
    }
}
ORDER BY ?año
"""

print("Materias con info completa:")
for fila in g.query(query_3):
    profesor = fila.profLabel if fila.profLabel else "(sin asignar)"
    print(f"  - {fila.label} (año {fila.año}) — dictada por {profesor}")

Materias con info completa:
  - Algoritmos y Estructuras de Datos (año 2) — dictada por Profesor Fulano
  - Microeconomía (año 2) — dictada por Profesor Mengano
  - Machine Learning (año 3) — dictada por Profesor Fulano
  - Bases de Datos (año 3) — dictada por Profesor Fulano
  - Procesamiento de Lenguaje Natural (año 4) — dictada por Luciano del Corro


In [9]:
# Query 4 — Estudiantes y qué materias de IA cursan
query_4 = """
PREFIX :     <http://example.org/udesa/>
PREFIX rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?estLabel ?matLabel ?año
WHERE {
    ?estudiante rdf:type :Estudiante ;
                rdfs:label ?estLabel ;
                :cursa ?materia .

    ?materia rdfs:label ?matLabel ;
             :perteneceA :IngenieriaIA ;
             :año ?año .
}
ORDER BY ?estLabel ?año
"""

print("Estudiantes y materias de IA que cursan:")
for fila in g.query(query_4):
    print(f"  - {fila.estLabel} cursa {fila.matLabel} (año {fila.año})")

Estudiantes y materias de IA que cursan:
  - Agustina Videla Rivero cursa Procesamiento de Lenguaje Natural (año 4)


In [10]:
# Query 5a — Correlativas DIRECTAS de NLP (1 salto, sin property path)
query_5a = """
PREFIX :     <http://example.org/udesa/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?correlativa ?label
WHERE {
    :NLP :tieneCorrelativa ?correlativa .
    ?correlativa rdfs:label ?label .
}
"""

print("Correlativas DIRECTAS de NLP (1 salto):")
for fila in g.query(query_5a):
    print(f"  - {fila.label}")

Correlativas DIRECTAS de NLP (1 salto):
  - Machine Learning


In [11]:
# Query 5b — TODAS las correlativas de NLP (transitivamente, con +)
query_5b = """
PREFIX :     <http://example.org/udesa/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?correlativa ?label
WHERE {
    :NLP :tieneCorrelativa+ ?correlativa .
    ?correlativa rdfs:label ?label .
}
"""

print("TODAS las correlativas de NLP (1 o más saltos):")
for fila in g.query(query_5b):
    print(f"  - {fila.label}")

TODAS las correlativas de NLP (1 o más saltos):
  - Machine Learning
  - Algoritmos y Estructuras de Datos


In [12]:
# Query 6 — Bonus: para cada estudiante, ¿qué prerrequisitos
# transitivos tienen las materias que cursan?
query_6 = """
PREFIX :     <http://example.org/udesa/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?estLabel ?matLabel ?prereqLabel
WHERE {
    ?estudiante rdfs:label ?estLabel ;
                :cursa ?materia .

    ?materia rdfs:label ?matLabel ;
             :tieneCorrelativa+ ?prereq .

    ?prereq rdfs:label ?prereqLabel .
}
ORDER BY ?estLabel ?matLabel
"""

print("Prerrequisitos transitivos por estudiante y materia:")
for fila in g.query(query_6):
    print(f"  - {fila.estLabel} cursa {fila.matLabel}, prereq: {fila.prereqLabel}")

Prerrequisitos transitivos por estudiante y materia:
  - Agustina Videla Rivero cursa Procesamiento de Lenguaje Natural, prereq: Machine Learning
  - Agustina Videla Rivero cursa Procesamiento de Lenguaje Natural, prereq: Algoritmos y Estructuras de Datos
